## Check CNA based classifications of arm level events
Use KM of centromere flanking normalization *k*-mers

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
class_df = pd.read_csv("/g/korbel/hain/centromere/PCAWG_classification.tsv", sep="\t")

In [ ]:
samples = list(class_df["SAMPLE"].unique())

## Get median coverage
Load normalization kmer slices and calulcate median KM for NORM_P and NORM_Q

In [ ]:
chromosomes = ["chr1"] + [f"chr{x}" for x in range(3, 13)] + [f"chr{x}" for x in range(16, 21)]

In [ ]:
series = []

for chrom in chromosomes:
    print(chrom)
    km_df = pd.read_csv(f"/scratch/hain/centromere/cancer/km_tables/{chrom}_PCAWG.kmer_km.NORM.tsv", sep="\t")
    
    for s in samples:
        if f"{s}_KM" not in km_df.columns:
            km_df[f"{s}_KM"] = 1
            
    p_median = km_df[km_df["CLUSTER"] == f"{chrom}_NORM_P"][[f"{x}_KM" for x in samples]].median(axis=0).rename(f"{chrom}_P_COV")
    q_median = km_df[km_df["CLUSTER"] == f"{chrom}_NORM_Q"][[f"{x}_KM" for x in samples]].median(axis=0).rename(f"{chrom}_Q_COV")
    
    series.append(p_median)
    series.append(q_median)

In [ ]:
median_df = pd.concat(series, axis=1)

In [ ]:
class_df = class_df[class_df["CHROM"].isin(chromosomes)]

In [ ]:
samplesheet = pd.read_csv("/g/korbel/hain/centromere/samplesheets/pcawg.tumor.normal.tsv", sep="\t")

In [ ]:
col_p_cov = []
col_q_cov = []
col_purity = []

for _, row in class_df.iterrows():
    col_p_cov.append(median_df.loc[f"{row['SAMPLE']}_KM"][f"{row['CHROM']}_P_COV"])
    col_q_cov.append(median_df.loc[f"{row['SAMPLE']}_KM"][f"{row['CHROM']}_Q_COV"])
    try:
        col_purity.append(samplesheet[samplesheet["pickedTumor"].eq(row["SAMPLE"])]["purity"].values[0])
    except IndexError:
        col_purity.append(None)
        
class_df["P_COV"] = col_p_cov
class_df["Q_COV"] = col_q_cov
class_df["PURITY"] = col_purity

In [ ]:
def get_expected_coverage_ratio (event:str, purity:float):
    if event == "P_ARM_LOSS" or event == "Q_ARM_LOSS":
        return (2 - purity) / 2
    elif event == "P_ARM_GAIN" or event == "Q_ARM_GAIN":
        return (2 + purity) / 2
    elif event == "ISO_P" or event == "ISO_Q":
        return (2 - purity) / (2 + purity)
    elif event in ["NORMAL", "WHOLE_LOSS", "WHOLE_GAIN", "GAIN_4_4"]:
        return purity - purity + 1
    else:
        return pd.NA

In [ ]:
def make_event_specific_coverage_ratio(event:str, p_cov:float, q_cov:float):
    try:
        if event == "P_ARM_LOSS":
            return p_cov / q_cov
        elif event == "Q_ARM_LOSS":
            return q_cov / p_cov
        elif event == "P_ARM_GAIN":
            return p_cov / q_cov
        elif event == "Q_ARM_GAIN":
            return q_cov / p_cov
        elif event == "ISO_Q":
            return p_cov / q_cov
        elif event == "ISO_P":
            return q_cov / p_cov
        elif event in ["NORMAL", "WHOLE_LOSS", "WHOLE_GAIN", "GAIN_4_4"]:
            return p_cov / q_cov
        else:
            return pd.NA
    except ZeroDivisionError:
        return pd.NA

In [ ]:
col_ratio = []
col_expected = []

for idx, row in class_df.iterrows():
    event = row["CN_CLASS"]
    p_cov = row["P_COV"]
    q_cov = row["Q_COV"]
    col_ratio.append(make_event_specific_coverage_ratio(event, p_cov, q_cov))
    col_expected.append(get_expected_coverage_ratio(event, row["PURITY"]))
    
class_df["COVERAGE_RATIO"] = col_ratio
class_df["EXPECTED_COVERAGE_RATIO"] = col_expected

In [ ]:
for event in ["P_ARM_LOSS", "Q_ARM_LOSS", "P_ARM_GAIN", "Q_ARM_GAIN", "ISO_Q", "ISO_P", "NORMAL"]:
    
    plot_df = class_df[class_df["CN_CLASS"].eq(event)]
   
    fig, ax = plt.subplots()
    sns.scatterplot(
        data=plot_df,
        x="PURITY",
        y="COVERAGE_RATIO",
        hue="CHROM",
        ax=ax
    )
    ax.set_title(event)
    ax.plot(np.linspace(0.2, 1, 100), get_expected_coverage_ratio(event, np.linspace(0.2, 1, 100)))
    #ax.fill_between(np.linspace(0.2, 1, 100), 0.6 * get_expected_coverage_ratio(event, np.linspace(0.2, 1, 100)), 1.4 * get_expected_coverage_ratio(event, np.linspace(0.2, 1, 100)), alpha=0.2, color='gray')
    ax.fill_between(np.linspace(0.2, 1, 100), (0.6 * get_expected_coverage_ratio(event, np.linspace(0.2, 1, 100)) - 1) + 1, 0.2 + get_expected_coverage_ratio(event, np.linspace(0.2, 1, 100)), alpha=0.2, color='gray')
    ax.set_yscale("log")
    plt.show()

In [ ]:
class_df["LOWER_EXPECTED_COVERAGE_RATIO"] = 0.6 * class_df["EXPECTED_COVERAGE_RATIO"]
class_df["UPPER_EXPECTED_COVERAGE_RATIO"] = 1.4 * class_df["EXPECTED_COVERAGE_RATIO"]
class_df["WITHIN_EXPECTED_RANGE"] = (class_df["COVERAGE_RATIO"] >= class_df["LOWER_EXPECTED_COVERAGE_RATIO"]) & (class_df["COVERAGE_RATIO"] <= class_df["UPPER_EXPECTED_COVERAGE_RATIO"])

In [ ]:
col_canonical_CN = []
for _, row in class_df.iterrows():
    if row["WITHIN_EXPECTED_RANGE"]:
        if row["CN_CLASS"] == "NORMAL":
            col_canonical_CN.append("2_2")
        elif row["CN_CLASS"] == "WHOLE_LOSS":
            col_canonical_CN.append("1_1")
        elif row["CN_CLASS"] == "WHOLE_GAIN":
            col_canonical_CN.append("3_3")
        elif row["CN_CLASS"] == "GAIN_4_4":
            col_canonical_CN.append("4_4")
        elif row["CN_CLASS"] == "P_ARM_LOSS":
            col_canonical_CN.append("1_2")
        elif row["CN_CLASS"] == "Q_ARM_LOSS":
            col_canonical_CN.append("2_1")
        elif row["CN_CLASS"] == "P_ARM_GAIN":
            col_canonical_CN.append("3_2")
        elif row["CN_CLASS"] == "Q_ARM_GAIN":
            col_canonical_CN.append("2_3")
        elif row["CN_CLASS"] == "ISO_P":
            col_canonical_CN.append("1_3")
        elif row["CN_CLASS"] == "ISO_Q":
            col_canonical_CN.append("3_1")
        else:
            col_canonical_CN.append("0_0")
    else:
        col_canonical_CN.append("0_0")
class_df["CANONICAL_CN"] = col_canonical_CN

### add CN information to cancer samplesheet

In [ ]:
from tqdm import tqdm

In [ ]:
chrom_cn_columns = {}
for chrom in chromosomes:
    chrom_cn_columns[chrom] = []
    
for _, row in tqdm(samplesheet.iterrows(), total=samplesheet.shape[0]):
    for chrom in chromosomes:
        tmp = class_df[class_df["SAMPLE"].eq(row["pickedTumor"]) & class_df["CHROM"].eq(chrom)]
        if tmp.shape[0] == 0:
            chrom_cn_columns[chrom].append("0_0")
        else:
            chrom_cn_columns[chrom].append(tmp["CANONICAL_CN"].values[0])
            
for chrom in chromosomes:
    samplesheet[f"{chrom}_CN"] = chrom_cn_columns[chrom]

In [ ]:
samplesheet.to_csv("/g/korbel/hain/centromere/samplesheets/pcawg.tumor.normal.CN.tsv", sep="\t", index=False)